# GPU G1.1 — Realistic parity and promotion closure

Final bounded Kaggle T4 audit. It proposes independent CLIP, translator, NVDEC-MB1, and NVDEC-neural promotion decisions without changing Stage1, Stage2, temporal research, or production AUTO policy.

Required Kaggle inputs (nested roots supported): raw AIC dataset, Stage1 index, Stage1B encoder report, Stage1E language freeze, Stage1C query bundle, RT2 benchmark bundle, offline OpenAI CLIP, offline OPUS vi-en, and optional PyNvVideoCodec CPython 3.12 wheel. Exact default mounts are printed by the next cell. Model execution is offline/local-only. Repository cloning is the only optional network action. Output: `/kaggle/working/triage_eg_gpu_g11_bundle.zip`.

In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, time
from dataclasses import replace
from datetime import UTC, datetime
import numpy as np

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
if not (REPO_DIR / 'src/triage_eg').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
WHEEL_INPUT = Path(os.environ.get('AIC_PYNVVIDEOCODEC_WHEEL_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-pynvvideocodec-wheel'))
wheel_candidates = sorted(WHEEL_INPUT.rglob('pynvvideocodec-*-cp312-*.whl')) if WHEEL_INPUT.exists() else []
if wheel_candidates:
    if sys.version_info[:2] != (3, 12): raise RuntimeError(f'PyNvVideoCodec wheel requires CPython 3.12, found {sys.version}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheel_candidates[0])], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
OUTPUT_ROOT = Path('/kaggle/working/triage_eg_gpu_g11')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def resolve_root(requested, marker, max_depth=8):
    requested, marker = Path(requested), Path(marker)
    if (requested / marker).is_file(): return requested.resolve()
    candidates = []
    for base in (requested, Path('/kaggle/input')):
        if not base.exists(): continue
        for found in base.rglob(marker.name):
            if not found.is_file() or len(found.relative_to(base).parts) > max_depth: continue
            root = found.parents[len(marker.parts) - 1]
            if (root / marker).is_file(): candidates.append(root.resolve())
    unique = sorted(set(candidates), key=lambda value: (len(value.parts), value.as_posix()))
    if not unique: raise FileNotFoundError(f'Cannot resolve {marker} below {requested}')
    return unique[0]

DEFAULTS = {
 'dataset': '/kaggle/input/datasets/nadkli/dataset-aic',
 'stage1': '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle',
 'stage1b': '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports',
 'stage1e': '/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze',
 'stage1c': '/kaggle/input/datasets/irthn1311/triage-eg-stage1c-qualitative-eval-bundle',
 'rt2': '/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle',
 'clip': '/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32',
 'opus': '/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en',
 'pynvvideocodec_wheel': str(WHEEL_INPUT),
}
DATA_ROOT = Path(os.environ.get('AIC_DATA_ROOT', DEFAULTS['dataset']))
STAGE1_ROOT = resolve_root(os.environ.get('AIC_STAGE1_ROOT', DEFAULTS['stage1']), 'index/clip_vectors.f16.npy')
STAGE1B_ROOT = resolve_root(os.environ.get('AIC_STAGE1B_ROOT', DEFAULTS['stage1b']), 'encoder/selected_encoder_contract.json')
STAGE1E_ROOT = resolve_root(os.environ.get('AIC_STAGE1E_ROOT', DEFAULTS['stage1e']), 'language_path_contract.json')
STAGE1C_ROOT = resolve_root(os.environ.get('AIC_STAGE1C_ROOT', DEFAULTS['stage1c']), 'query_suite/query_suite.jsonl')
RT2_ROOT = resolve_root(os.environ.get('AIC_RT2_ROOT', DEFAULTS['rt2']), 'rt2_ai_benchmark.jsonl')
CLIP_ROOT = resolve_root(os.environ.get('AIC_CLIP_ROOT', DEFAULTS['clip']), 'checkpoint/ViT-B-32.pt')
OPUS_ROOT = resolve_root(os.environ.get('AIC_OPUS_ROOT', DEFAULTS['opus']), 'model/config.json')
INPUTS = {**DEFAULTS, 'resolved': {'dataset': str(DATA_ROOT), 'stage1': str(STAGE1_ROOT), 'stage1b': str(STAGE1B_ROOT), 'stage1e': str(STAGE1E_ROOT), 'stage1c': str(STAGE1C_ROOT), 'rt2': str(RT2_ROOT), 'clip': str(CLIP_ROOT), 'opus': str(OPUS_ROOT)}, 'repository': str(REPO_DIR)}
print(json.dumps({'commit': COMMIT, 'inputs': INPUTS, 'output_zip': '/kaggle/working/triage_eg_gpu_g11_bundle.zip'}, indent=2))

In [ ]:
import torch
from triage_eg.video import HardwareConfig, nvdec_preflight, resolve_hardware
from triage_eg.video.g1_audit import representative_videos
from triage_eg.video.g11_audit import load_frozen_query_suite

if not torch.cuda.is_available(): raise RuntimeError('GPU_G11_REQUIRES_KAGGLE_T4_CUDA')
NVDEC_PREFLIGHT = nvdec_preflight()
AUTO_POLICY = resolve_hardware(HardwareConfig(), torch_module=torch, nvdec_probe=NVDEC_PREFLIGHT).as_dict()
GPU_PREFLIGHT = {'python': platform.python_version(), 'torch': torch.__version__, 'torch_cuda': getattr(torch.version, 'cuda', None), 'cuda_available': True, 'gpu_name': torch.cuda.get_device_name(0), 'nvdec': NVDEC_PREFLIGHT, 'effective_auto_before_audit': AUTO_POLICY, 'wheel': str(wheel_candidates[0]) if wheel_candidates else None}
QUERY_ROWS = load_frozen_query_suite(STAGE1C_ROOT / 'query_suite/query_suite.jsonl', RT2_ROOT / 'rt2_ai_benchmark.jsonl', maximum=100)
VIDEOS = representative_videos(DATA_ROOT, limit=int(os.environ.get('AIC_G11_VIDEO_LIMIT', '4')))
if not 3 <= len(VIDEOS) <= 6: raise RuntimeError(f'Expected 3-6 representative videos, found {len(VIDEOS)}')
INPUTS['resolved']['videos'] = [str(path) for path in VIDEOS]
print(json.dumps({'preflight': GPU_PREFLIGHT, 'queries': len(QUERY_ROWS), 'videos': INPUTS['resolved']['videos']}, indent=2))

In [ ]:
from triage_eg.video.g11_audit import benchmark_mb1_workload
MB1_BENCHMARK, NVDEC_MB1_PARITY = benchmark_mb1_workload(VIDEOS, nvdec_available=bool(NVDEC_PREFLIGHT['available']))
print(json.dumps({'samples_per_second': MB1_BENCHMARK['samples_per_second'], 'records': len(MB1_BENCHMARK['records']), 'status': NVDEC_MB1_PARITY['status']}, indent=2))

In [ ]:
from triage_eg.video.g11_audit import benchmark_m1_local_workload
M1_BENCHMARK, M1_NVDEC_PARITY, CPU_DENSE_IMAGES, NVDEC_DENSE_IMAGES = benchmark_m1_local_workload(VIDEOS, nvdec_available=bool(NVDEC_PREFLIGHT['available']))
print(json.dumps({'workload': M1_BENCHMARK['workload'], 'records': len(M1_BENCHMARK['records']), 'parity_rows': len(M1_NVDEC_PARITY['rows'])}, indent=2))

In [ ]:
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, QueryRequest, config_from_yaml
from triage_eg.video.g11_audit import embedding_parity, retrieval_agreement

base_kwargs = dict(stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT, clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT, stage1d_config=REPO_DIR / 'configs/retrieval/stage1d_translation_ablation.yaml', build_git_commit=COMMIT)
SHARED_RUNTIME_ROOT = OUTPUT_ROOT / '_shared_stage2_runtime'
cpu_config = config_from_yaml(REPO_DIR / 'configs/retrieval/stage2_operational_runtime.yaml', output_root=SHARED_RUNTIME_ROOT / 'cpu', **base_kwargs)
gpu_base = config_from_yaml(REPO_DIR / 'configs/retrieval/stage2_operational_runtime_gpu.yaml', output_root=SHARED_RUNTIME_ROOT / 'gpu', **base_kwargs)
gpu_config = replace(gpu_base, hardware_mode='gpu', video_backend='opencv')
requests = [QueryRequest(row['query_id'], row['text'], row['language'], 100) for row in QUERY_ROWS]
cpu_runtime = OperationalRetrievalRuntime(cpu_config).load()
started = time.monotonic(); cpu_batch = cpu_runtime.encode_requests(requests); CPU_QUERY_MS = (time.monotonic() - started) * 1000
gpu_runtime = OperationalRetrievalRuntime(gpu_config).load()
started = time.monotonic(); gpu_batch = gpu_runtime.encode_requests(requests); GPU_END_TO_END_MS = (time.monotonic() - started) * 1000
frozen_clip_inputs = [row['clip_input_text'] for row in cpu_batch.encodings]
started = time.monotonic(); gpu_embeddings = gpu_runtime.encoder.encode_text(frozen_clip_inputs); GPU_QUERY_MS = (time.monotonic() - started) * 1000
INDEX_VECTORS = np.load(STAGE1_ROOT / 'index/clip_vectors.f16.npy', mmap_mode='r', allow_pickle=False)
INDEX_NORMS = np.load(STAGE1_ROOT / 'index/vector_norms.f32.npy', mmap_mode='r', allow_pickle=False)
CLIP_EMBEDDING_PARITY = embedding_parity(cpu_batch.embeddings, gpu_embeddings)
CLIP_RETRIEVAL_PARITY = retrieval_agreement(cpu_batch.embeddings, gpu_embeddings, INDEX_VECTORS, index_norms=INDEX_NORMS, top_k=100)
print(json.dumps({'embedding': CLIP_EMBEDDING_PARITY, 'retrieval': {key: CLIP_RETRIEVAL_PARITY[key] for key in ('query_count', 'top1_exact_matches', 'top1_changes', 'overlap', 'mean_rank_displacement', 'median_rank_displacement', 'maximum_rank_displacement', 'exact_top50_order_matches', 'exact_top50_order_is_hard_gate', 'status')}}, indent=2))

In [ ]:
from triage_eg.video.g11_audit import benchmark_clip_batches, consumer_specific_nvdec_verdicts, in_memory_path_parity

CLIP_BATCH_BENCHMARK = benchmark_clip_batches(cpu_runtime.encoder, gpu_runtime.encoder, CPU_DENSE_IMAGES, batch_sizes=(1, 8, 16, 32, 64))
M1_IN_MEMORY_PARITY = in_memory_path_parity(cpu_runtime.encoder, CPU_DENSE_IMAGES)
if NVDEC_PREFLIGHT['available'] and NVDEC_DENSE_IMAGES:
    cpu_decoded_embeddings = gpu_runtime.encoder.encode_rgb_arrays(CPU_DENSE_IMAGES)
    nvdec_decoded_embeddings = gpu_runtime.encoder.encode_rgb_arrays(NVDEC_DENSE_IMAGES)
    NVDEC_NEURAL_EMBEDDING = embedding_parity(cpu_decoded_embeddings, nvdec_decoded_embeddings)
    NVDEC_NEURAL_EMBEDDING['rows'] = M1_NVDEC_PARITY['rows']
    NVDEC_NEURAL_RETRIEVAL = retrieval_agreement(cpu_decoded_embeddings, nvdec_decoded_embeddings, INDEX_VECTORS, index_norms=INDEX_NORMS, top_k=100)
else:
    NVDEC_NEURAL_EMBEDDING = {'status': 'UNAVAILABLE', 'rows': M1_NVDEC_PARITY['rows']}
    NVDEC_NEURAL_RETRIEVAL = {'status': 'UNAVAILABLE'}
NVDEC_STATUSES = consumer_specific_nvdec_verdicts(NVDEC_MB1_PARITY, NVDEC_NEURAL_EMBEDDING, NVDEC_NEURAL_RETRIEVAL, nvdec_available=bool(NVDEC_PREFLIGHT['available']))
NVDEC_NEURAL_PARITY = {'status': NVDEC_STATUSES['NVDEC_NEURAL'], 'embedding': NVDEC_NEURAL_EMBEDDING, 'retrieval': NVDEC_NEURAL_RETRIEVAL, 'decoder_rows': M1_NVDEC_PARITY['rows']}
print(json.dumps({'clip_batches': CLIP_BATCH_BENCHMARK, 'm1_in_memory': M1_IN_MEMORY_PARITY, 'nvdec': NVDEC_STATUSES}, indent=2))

In [ ]:
from triage_eg.video.g11_audit import build_promotion_policy, clip_gpu_verdict

sanity_indices = [index for index, row in enumerate(QUERY_ROWS) if row['language'] == 'vi'][:5]
cpu_texts = [cpu_batch.encodings[index]['clip_input_text'] for index in sanity_indices]
gpu_texts = [gpu_batch.encodings[index]['clip_input_text'] for index in sanity_indices]
TRANSLATOR_GPU_STATUS = 'KEEP' if sanity_indices and cpu_texts == gpu_texts else 'DROP'
TRANSLATOR_SANITY = {'status': TRANSLATOR_GPU_STATUS, 'query_count': len(sanity_indices), 'translated_text_for_clip_equal': cpu_texts == gpu_texts, 'cpu': cpu_texts, 'gpu': gpu_texts, 'cpu_manifest': cpu_runtime.translator.runtime_manifest(), 'gpu_manifest': gpu_runtime.translator.runtime_manifest()}
CLIP_GPU_STATUS = clip_gpu_verdict(CLIP_EMBEDDING_PARITY, CLIP_RETRIEVAL_PARITY)
RUN_ID = 'gpu_g11_' + datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
POLICY = build_promotion_policy(clip_status=CLIP_GPU_STATUS, translator_status=TRANSLATOR_GPU_STATUS, nvdec_mb1_status=NVDEC_STATUSES['NVDEC_MB1'], nvdec_neural_status=NVDEC_STATUSES['NVDEC_NEURAL'], build_commit=COMMIT, evidence_run_id=RUN_ID)
print(json.dumps({'translator': TRANSLATOR_SANITY, 'clip_status': CLIP_GPU_STATUS, 'policy': POLICY}, indent=2))

In [ ]:
CPU_MANIFEST, GPU_MANIFEST = cpu_runtime.runtime_manifest(), gpu_runtime.runtime_manifest()
cpu_runtime.close(); gpu_runtime.close()
GPU_INFRASTRUCTURE = 'FROZEN_READY' if TRANSLATOR_GPU_STATUS == 'KEEP' and CLIP_GPU_STATUS in {'KEEP', 'CONDITIONAL'} and M1_IN_MEMORY_PARITY['status'] == 'PASS' else 'BLOCKED'
FINAL = {'GPU_G11_REAL_STATUS': 'COMPLETE', 'CPU_FALLBACK': 'KEEP', 'OPTIMIZED_OPENCV': 'KEEP', 'TRANSLATOR_GPU': TRANSLATOR_GPU_STATUS, 'CLIP_GPU': CLIP_GPU_STATUS, 'NVDEC_MB1': NVDEC_STATUSES['NVDEC_MB1'], 'NVDEC_NEURAL': NVDEC_STATUSES['NVDEC_NEURAL'], 'M1_IN_MEMORY_CLIP': 'KEEP' if M1_IN_MEMORY_PARITY['status'] == 'PASS' else 'DROP', 'GPU_INFRASTRUCTURE': GPU_INFRASTRUCTURE, 'RETURN_TO_MAIN_PIPELINE': 'YES' if GPU_INFRASTRUCTURE == 'FROZEN_READY' else 'NO'}
PERFORMANCE = {'clip_query': {'count': len(QUERY_ROWS), 'cpu_end_to_end_ms': CPU_QUERY_MS, 'gpu_end_to_end_ms': GPU_END_TO_END_MS, 'gpu_frozen_clip_input_ms': GPU_QUERY_MS}, 'clip_batches': CLIP_BATCH_BENCHMARK, 'mb1': MB1_BENCHMARK, 'm1': M1_BENCHMARK}
ISSUES = list(CLIP_BATCH_BENCHMARK['issues'])
if not NVDEC_PREFLIGHT['available']: ISSUES.append({'severity': 'INFO', 'code': 'NVDEC_UNAVAILABLE', 'detail': NVDEC_PREFLIGHT.get('reason')})
if CLIP_GPU_STATUS == 'CONDITIONAL': ISSUES.append({'severity': 'WARNING', 'code': 'CLIP_GPU_CONDITIONAL'})
RUN_MANIFEST = {'sprint': 'GPU_G1.1', 'run_id': RUN_ID, 'created_at': datetime.now(UTC).isoformat(), 'git_commit': COMMIT, 'inputs': INPUTS, 'hardware': GPU_PREFLIGHT, 'query_count': len(QUERY_ROWS), 'runtime_manifests': {'cpu': CPU_MANIFEST, 'gpu': GPU_MANIFEST}, 'frozen_contracts': {'stage1': 'EXACT_NUMPY_CPU_UNCHANGED', 'stage2_semantics': 'UNCHANGED', 'temporal_research': 'UNCHANGED', 'mb1_signal_thresholds': 'UNCHANGED'}, 'final_status': FINAL}
print(json.dumps(FINAL, indent=2))

In [ ]:
from triage_eg.video.g11_audit import write_g11_bundle
ARTIFACTS = {'gpu_preflight.json': GPU_PREFLIGHT, 'mb1_decoder_benchmark.json': MB1_BENCHMARK, 'm1_decoder_benchmark.json': M1_BENCHMARK, 'nvdec_mb1_parity.json': NVDEC_MB1_PARITY, 'nvdec_neural_parity.json': NVDEC_NEURAL_PARITY, 'clip_embedding_parity.json': {**CLIP_EMBEDDING_PARITY, 'm1_in_memory_path_parity': M1_IN_MEMORY_PARITY}, 'clip_retrieval_parity.json': CLIP_RETRIEVAL_PARITY, 'clip_batch_benchmark.json': CLIP_BATCH_BENCHMARK, 'translator_gpu_sanity.json': TRANSLATOR_SANITY, 'gpu_promotion_policy.json': POLICY, 'performance_summary.json': PERFORMANCE, 'run_manifest.json': RUN_MANIFEST, 'issues.jsonl': ISSUES}
ZIP_PATH = write_g11_bundle(OUTPUT_ROOT, ARTIFACTS)
for key, value in FINAL.items(): print(f'{key}={value}')
print('DOWNLOAD_ZIP=' + str(ZIP_PATH))